# Module 5: Multi-Mode Optics, Linear Networks & CV Quantum Information
## Notebook 5.1: The Quantum Beam Splitter & Entanglement Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_5_multimode_quantum_information/01_quantum_beam_splitter.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** Open Quantum Systems, Dissipation & Trajectories
* **Module 5:** **Multi-Mode Optics, Linear Networks & CV Quantum Information**
  * **5.1 The Quantum Beam Splitter & Entanglement Generation** $\leftarrow$ *(You are here)*
  * **5.2** Two-Photon Interference & The Hong-Ou-Mandel Effect
  * **5.3** Two-Mode Squeezing, EPR States & Continuous-Variable Inseparability
  * **5.4** Continuous-Variable Quantum Teleportation (Braunstein-Kimble)
  * **5.5** Gaussian States, Symplectic Operations & Covariance Matrices
  * **5.6** Quantum State Engineering: Photon Addition, Subtraction & Catalysis
  * **5.7** Linear Optical Quantum Computing & Boson Sampling
* **Module 6:** Advanced Frontiers & Modern Computational Quantum Optics

---

### Learning Objectives
1. **The Quantum Beam Splitter Transformation:** Formulate the unitary evolution operator $\hat{U}_{\text{BS}}(\theta) = \exp\left[\theta(\hat{a}^\dagger\hat{b} - \hat{a}\hat{b}^\dagger)\right]$ and its linear action on field annihilation operators.
2. **Coherent State Separability:** Prove analytically and demonstrate numerically that coherent states incident on a beam splitter split classically without generating entanglement: $|\alpha\rangle|0\rangle \to |\alpha\cos\theta\rangle|-\alpha\sin\theta\rangle$.
3. **Single-Photon Path Entanglement:** Analyze Fock state splitting and show that a single photon $|1\rangle|0\rangle$ creates a maximally entangled Bell-like state $\frac{|1,0\rangle + |0,1\rangle}{\sqrt{2}}$ with entanglement entropy $S = \ln 2$.
4. **Binomial State Generation:** Compute the joint photon statistics $P(n_a, n_b)$ for multi-photon Fock inputs $|n\rangle|0\rangle$ and verify total photon number conservation $[\hat{U}_{\text{BS}}, \hat{N}_a + \hat{N}_b] = 0$.
5. **Entanglement as a Non-Classicality Witness:** Explore the Kim-Son-Buzek-Knight theorem: a beam splitter acts as an entanglement gate if and only if the input state possesses quantum non-classicality.



In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.special import binom
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

# Verify environment capabilities
print(f"NumPy version: {np.__version__}")
try:
    import qutip as qt
    print(f"QuTiP version: {qt.__version__} (Available for optional comparison)")
    HAS_QUTIP = True
except ImportError:
    print("QuTiP not installed; running pure, optimized native NumPy/SciPy engine.")
    HAS_QUTIP = False



---

## Theoretical Framework: The Quantum Beam Splitter

### 1. The Beam Splitter as an $SU(2)$ Transformation
In classical optics, a lossless dielectric beam splitter relates the input electric field amplitudes $(E_{a,\text{in}}, E_{b,\text{in}})^T$ to output amplitudes $(E_{a,\text{out}}, E_{b,\text{out}})^T$ via a $2 \times 2$ unitary scattering matrix.
In quantum optics, energy conservation and the preservation of canonical commutation relations:
$$[\hat{a}_{\text{out}}, \hat{a}_{\text{out}}^\dagger] = 1, \quad [\hat{b}_{\text{out}}, \hat{b}_{\text{out}}^\dagger] = 1, \quad [\hat{a}_{\text{out}}, \hat{b}_{\text{out}}^\dagger] = 0$$
demand that the mode annihilation operators transform unitarily:
$$\begin{pmatrix} \hat{a}_{\text{out}} \\ \hat{b}_{\text{out}} \end{pmatrix} = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} \hat{a}_{\text{in}} \\ \hat{b}_{\text{in}} \end{pmatrix}$$
where $\theta$ is the beam splitter angle, related to transmittance $T$ and reflectance $R$ by:
$$T = \cos^2\theta, \quad R = \sin^2\theta, \quad T + R = 1$$
For a balanced **50:50 beam splitter**, $\theta = \pi/4$, yielding $T = R = 1/2$.

In the Schrödinger picture, this transformation is generated by the unitary operator $\hat{U}_{\text{BS}}(\theta)$:
$$\hat{U}_{\text{BS}}(\theta) = \exp\left[ \theta (\hat{a}^\dagger \hat{b} - \hat{a}\hat{b}^\dagger) \right]$$
which satisfies:
$$\hat{a}_{\text{out}} = \hat{U}_{\text{BS}}^\dagger \hat{a} \hat{U}_{\text{BS}} = \hat{a}\cos\theta + \hat{b}\sin\theta$$
$$\hat{b}_{\text{out}} = \hat{U}_{\text{BS}}^\dagger \hat{b} \hat{U}_{\text{BS}} = -\hat{a}\sin\theta + \hat{b}\cos\theta$$

---

### 2. Symmetries & Total Photon Number Conservation
The generator $\hat{G} = -i(\hat{a}^\dagger\hat{b} - \hat{a}\hat{b}^\dagger)$ belongs to the Lie algebra of $\mathfrak{su}(2)$, mapped via the Schwinger representation:
$$\hat{J}_x = \frac{\hat{a}^\dagger\hat{b} + \hat{b}^\dagger\hat{a}}{2}, \quad \hat{J}_y = \frac{\hat{a}^\dagger\hat{b} - \hat{b}^\dagger\hat{a}}{2i}, \quad \hat{J}_z = \frac{\hat{a}^\dagger\hat{a} - \hat{b}^\dagger\hat{b}}{2}$$
The Casimir invariant of this algebra is the **total photon number operator**:
$$\hat{N}_{\text{tot}} = \hat{a}^\dagger\hat{a} + \hat{b}^\dagger\hat{b} = 2\hat{J}$$
Because $[\hat{U}_{\text{BS}}(\theta), \hat{N}_{\text{tot}}] = 0$, **the beam splitter strictly conserves the total photon number**! An input state with $N$ total photons will exclusively scatter into output states with exactly $N$ total photons:
$$\hat{U}_{\text{BS}} |n_a, n_b\rangle = \sum_{k=0}^{n_a + n_b} C_k |k, n_a + n_b - k\rangle$$

---

### 3. Coherent State Splitting: Classical Separability
Consider a coherent state $|\alpha\rangle_a$ incident on port $a$ and vacuum $|0\rangle_b$ on port $b$:
$$|\alpha\rangle_a |0\rangle_b = \hat{D}_a(\alpha)|0, 0\rangle = \exp\left(\alpha \hat{a}^\dagger - \alpha^* \hat{a}\right)|0, 0\rangle$$

Applying the beam splitter unitary:
$$\hat{U}_{\text{BS}} \hat{a}^\dagger \hat{U}_{\text{BS}}^\dagger = \hat{a}^\dagger \cos\theta - \hat{b}^\dagger \sin\theta$$
$$\hat{U}_{\text{BS}} |\alpha\rangle_a |0\rangle_b = \exp\left[ \alpha(\hat{a}^\dagger \cos\theta - \hat{b}^\dagger \sin\theta) - \text{h.c.} \right] |0, 0\rangle$$
$$\hat{U}_{\text{BS}} |\alpha\rangle_a |0\rangle_b = |\alpha\cos\theta\rangle_a \otimes |-\alpha\sin\theta\rangle_b$$

#### Fundamental Theorem of Coherent Splitting:
The output state remains an **exact product state of two independent coherent states**!
* **No entanglement** is generated ($S(\hat{\rho}_a) \equiv 0$).
* The optical power splits according to classical intensity laws: $\langle\hat{n}_a\rangle = |\alpha|^2 \cos^2\theta$, $\langle\hat{n}_b\rangle = |\alpha|^2 \sin^2\theta$.
* Coherent states are the **only pure states** that do not generate entanglement when mixed with vacuum at a beam splitter.

---

### 4. Fock State Splitting & Path Entanglement
By contrast, consider non-classical Fock states:

#### A. Single Photon Input $|1, 0\rangle$:
$$\hat{U}_{\text{BS}} |1, 0\rangle = \hat{U}_{\text{BS}} \hat{a}^\dagger |0, 0\rangle = (\hat{a}^\dagger \cos\theta - \hat{b}^\dagger \sin\theta)|0, 0\rangle = \cos\theta |1, 0\rangle - \sin\theta |0, 1\rangle$$
For a 50:50 beam splitter ($\theta = \pi/4$):
$$|\psi_{\text{out}}\rangle = \frac{|1, 0\rangle - |0, 1\rangle}{\sqrt{2}}$$
This is a **maximally entangled single-photon path state**! 
Tracing out mode $b$ leaves mode $a$ in the completely mixed statistical ensemble $\hat{\rho}_a = \frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1|$, with von Neumann entanglement entropy:
$$S(\hat{\rho}_a) = -\text{Tr}(\hat{\rho}_a \ln \hat{\rho}_a) = \ln 2 \approx 0.69315 \text{ nats}$$

#### B. Multi-Photon Input $|n, 0\rangle$ (Binomial State):
$$\hat{U}_{\text{BS}} |n, 0\rangle = \frac{1}{\sqrt{n!}} (\hat{a}^\dagger \cos\theta - \hat{b}^\dagger \sin\theta)^n |0, 0\rangle = \sum_{k=0}^n \sqrt{\binom{n}{k}} (\cos\theta)^k (-\sin\theta)^{n-k} |k, n-k\rangle$$
The photon probability distribution at port $a$ is strictly **binomial**:
$$P(n_a = k) = \binom{n}{k} T^k (1-T)^{n-k}$$



In [ ]:
# Implementation: High-Precision Quantum Beam Splitter Engine

class QuantumBeamSplitter:
    """
    Simulation engine for two-mode quantum beam splitters in the Fock basis.
    Operates on bipartite Hilbert spaces H_a (x) H_b truncated at dimension cutoff.
    """
    def __init__(self, n_cutoff=20):
        self.cutoff = int(n_cutoff)
        self.dim = self.cutoff * self.cutoff
        
        # Single-mode operators
        a_single = np.zeros((self.cutoff, self.cutoff), dtype=complex)
        for n in range(1, self.cutoff):
            a_single[n-1, n] = np.sqrt(n)
        Id_single = np.eye(self.cutoff, dtype=complex)
        
        # Two-mode operators: mode a and mode b
        # State ordering: |n_a, n_b> with index n_a * cutoff + n_b
        self.a = np.kron(a_single, Id_single)
        self.b = np.kron(Id_single, a_single)
        self.a_dag = self.a.conj().T
        self.b_dag = self.b.conj().T
        
        # Total photon number operator: N_tot = a^dag a + b^dag b
        self.N_tot = np.dot(self.a_dag, self.a) + np.dot(self.b_dag, self.b)
        
        # Beam splitter generator: G = a^dag b - a b^dag (anti-Hermitian)
        self.G = np.dot(self.a_dag, self.b) - np.dot(self.a, self.b_dag)

    def state_fock(self, na, nb):
        """Construct pure two-mode product Fock state |na, nb>."""
        psi = np.zeros(self.dim, dtype=complex)
        idx = na * self.cutoff + nb
        psi[idx] = 1.0
        return psi

    def state_coherent_vacuum(self, alpha):
        """Construct pure state |alpha>_a (x) |0>_b."""
        psi_a = np.zeros(self.cutoff, dtype=complex)
        alpha_val = complex(alpha)
        for n in range(self.cutoff):
            psi_a[n] = np.exp(-0.5 * np.abs(alpha_val)**2) * (alpha_val**n) / np.sqrt(float(math.factorial(n)))
        psi_a /= la.norm(psi_a)
        
        psi_b = np.zeros(self.cutoff, dtype=complex)
        psi_b[0] = 1.0  # Vacuum in mode b
        
        return np.kron(psi_a, psi_b)

    def unitary(self, theta):
        """Compute the beam splitter unitary matrix U_BS(theta) = exp(theta * G)."""
        return la.expm(float(theta) * self.G)

    def transform_state(self, psi_in, theta):
        """Propagate state vector through beam splitter with angle theta."""
        U = self.unitary(theta)
        return np.dot(U, psi_in)

    def joint_photon_distribution(self, psi):
        """Extract 2D joint probability distribution P(na, nb)."""
        probs = np.abs(psi)**2
        return probs.reshape((self.cutoff, self.cutoff))

    def partial_trace_a(self, psi):
        """Compute reduced density matrix of mode a: rho_a = Tr_b(|psi><psi|)."""
        psi_mat = psi.reshape((self.cutoff, self.cutoff))
        # rho_a[n, m] = sum_k psi_mat[n, k] * conj(psi_mat[m, k])
        rho_a = np.dot(psi_mat, psi_mat.conj().T)
        return rho_a

    def entanglement_entropy(self, psi):
        """Compute von Neumann entanglement entropy S(rho_a) = -Tr(rho_a ln rho_a)."""
        rho_a = self.partial_trace_a(psi)
        evals = np.real(la.eigvalsh(rho_a))
        # Filter out zero eigenvalues for numerical stability
        pos_evals = evals[evals > 1e-15]
        return -float(np.sum(pos_evals * np.log(pos_evals)))

print("QuantumBeamSplitter engine successfully loaded.")



In [ ]:
# Publication-Quality 4-Panel Visualization: Quantum Beam Splitter Entanglement & Statistics

bs = QuantumBeamSplitter(n_cutoff=16)

# 1. Binomial state splitting for Fock |4, 0> on 50:50 beam splitter (theta = pi/4)
theta_5050 = np.pi / 4.0
psi_fock4 = bs.state_fock(4, 0)
psi_out_fock4 = bs.transform_state(psi_fock4, theta_5050)
P_joint_fock4 = bs.joint_photon_distribution(psi_out_fock4)

# 2. Entanglement entropy S(theta) vs. beam splitter angle theta for various inputs
theta_vals = np.linspace(0.0, np.pi / 2.0, 60)
S_fock1 = []
S_fock2 = []
S_fock4 = []
S_coherent = []

psi_coh_in = bs.state_coherent_vacuum(alpha=2.0)

for th in theta_vals:
    U_th = bs.unitary(th)
    # Fock |1,0>
    out1 = np.dot(U_th, bs.state_fock(1, 0))
    S_fock1.append(bs.entanglement_entropy(out1))
    # Fock |2,0>
    out2 = np.dot(U_th, bs.state_fock(2, 0))
    S_fock2.append(bs.entanglement_entropy(out2))
    # Fock |4,0>
    out4 = np.dot(U_th, bs.state_fock(4, 0))
    S_fock4.append(bs.entanglement_entropy(out4))
    # Coherent |alpha=2, 0>
    out_coh = np.dot(U_th, psi_coh_in)
    S_coherent.append(bs.entanglement_entropy(out_coh))

# 3. Output photon statistics at port a: Fock |4, 0> vs. Coherent |alpha=2, 0>
P_a_fock4 = np.sum(P_joint_fock4, axis=1)

psi_out_coh = bs.transform_state(psi_coh_in, theta_5050)
P_joint_coh = bs.joint_photon_distribution(psi_out_coh)
P_a_coh = np.sum(P_joint_coh, axis=1)

# Analytical binomial distribution for |4, 0> on 50:50 BS: P(k) = binom(4, k) * (0.5)^4
k_vec = np.arange(5)
P_binomial_analytic = binom(4, k_vec) * (0.5**4)

# Generate 4-Panel Masterpiece Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------
# Panel (a): Joint Photon Number Distribution P(na, nb)
# -------------------------------------------------------------
ax_a = axes[0, 0]
im = ax_a.imshow(P_joint_fock4[:7, :7], cmap="magma", origin="lower", extent=[-0.5, 6.5, -0.5, 6.5])
cbar = plt.colorbar(im, ax=ax_a, fraction=0.046, pad=0.04)
cbar.set_label("Probability $P(n_a, n_b)$")

# Annotate non-zero diagonal entries along total photon conservation line na + nb = 4
for na in range(5):
    nb = 4 - na
    p_val = P_joint_fock4[na, nb]
    ax_a.text(nb, na, f"{p_val:.3f}", ha="center", va="center", color="cyan", fontweight="bold", fontsize=10)

ax_a.set_title(r"(a) Joint Distribution $P(n_a, n_b)$ for $|4, 0\rangle$ at 50:50 BS", fontweight="bold")
ax_a.set_xlabel(r"Output Port $b$ Photon Number $n_b$")
ax_a.set_ylabel(r"Output Port $a$ Photon Number $n_a$")
ax_a.plot([-0.5, 4.5], [4.5, -0.5], color="cyan", linestyle="--", linewidth=1.5, label=r"$n_a + n_b = 4$ Conservation")
ax_a.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (b): Entanglement Entropy vs. Beam Splitter Angle
# -------------------------------------------------------------
ax_b = axes[0, 1]
ax_b.plot(theta_vals / np.pi, S_fock1, color="#3498db", linewidth=2.2, label=r"Fock $|1, 0\rangle$ (Single-Photon)")
ax_b.plot(theta_vals / np.pi, S_fock2, color="#2ecc71", linewidth=2.2, label=r"Fock $|2, 0\rangle$")
ax_b.plot(theta_vals / np.pi, S_fock4, color="#e74c3c", linewidth=2.4, label=r"Fock $|4, 0\rangle$")
ax_b.plot(theta_vals / np.pi, S_coherent, color="black", linestyle="--", linewidth=2.0, label=r"Coherent $|\alpha=2, 0\rangle$ ($S \equiv 0$)")

ax_b.axvline(0.25, color="gray", linestyle=":", linewidth=1.5, label=r"50:50 Splitter ($\theta = \pi/4$)")
ax_b.axhline(np.log(2.0), color="#3498db", linestyle=":", alpha=0.7)
ax_b.annotate(r"Bell State: $S = \ln 2 \approx 0.693$", xy=(0.25, np.log(2.0)), xytext=(0.04, 0.85),
              arrowprops=dict(arrowstyle="->", color="#3498db", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#ebf5fb", ec="#3498db", lw=1.2),
              fontweight="bold")

ax_b.set_title(r"(b) Entanglement Entropy $S(\hat{\rho}_a)$ vs. Angle $\theta$", fontweight="bold")
ax_b.set_xlabel(r"Beam Splitter Angle $\theta / \pi$")
ax_b.set_ylabel(r"Von Neumann Entropy $S(\hat{\rho}_a)$ (nats)")
ax_b.set_ylim(-0.05, 1.6)
ax_b.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (c): Output Port a Statistics: Binomial vs. Poisson
# -------------------------------------------------------------
ax_c = axes[1, 0]
n_bins = np.arange(8)
ax_c.bar(n_bins[:5] - 0.15, P_a_fock4[:5], width=0.3, color="#e74c3c", alpha=0.85, label=r"Fock $|4, 0\rangle$ Output")
ax_c.plot(k_vec - 0.15, P_binomial_analytic, "kd", markersize=7, label=r"Analytic Binomial $B(4, 0.5)$")
ax_c.bar(n_bins + 0.15, P_a_coh[:8], width=0.3, color="#3498db", alpha=0.85, label=r"Coherent $|\alpha=2, 0\rangle$ Output")

ax_c.set_title(r"(c) Port $a$ Marginal Statistics: Binomial vs. Coherent", fontweight="bold")
ax_c.set_xlabel(r"Measured Photons $n_a$")
ax_c.set_ylabel(r"Probability $P(n_a)$")
ax_c.set_xlim(-0.6, 7.5)
ax_c.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (d): Entanglement Hierarchy: Fock Number Scaling
# -------------------------------------------------------------
ax_d = axes[1, 1]
fock_n_list = np.arange(1, 9)
S_max_fock = []

for n_in in fock_n_list:
    psi_n = bs.state_fock(n_in, 0)
    out_n = bs.transform_state(psi_n, theta_5050)
    S_max_fock.append(bs.entanglement_entropy(out_n))

ax_d.plot(fock_n_list, S_max_fock, "o-", color="#8e44ad", linewidth=2.4, markersize=8, label=r"50:50 BS Entanglement $S(n)$")
# Theoretical bound S_max = ln(n + 1)
theoretical_bound = np.log(fock_n_list + 1)
ax_d.plot(fock_n_list, theoretical_bound, "k--", linewidth=1.8, label=r"Maximally Entangled Cap $\ln(n + 1)$")

ax_d.set_title(r"(d) Maximum Entanglement Scaling with Input Photons $n$", fontweight="bold")
ax_d.set_xlabel(r"Input Fock Number $n$ in Mode $a$")
ax_d.set_ylabel(r"Peak Entanglement Entropy $S_{\rm max}$ (nats)")
ax_d.legend(loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()



> **Physical Insight: Why Does a Classical Beam Splitter Create Quantum Entanglement?**
>
> 1. **The Kim-Son-Buzek-Knight Theorem (2002):** A linear, passive beam splitter $\hat{U}_{\text{BS}}$ cannot create non-classicality on its own. If the input states are classical (i.e., mixtures of coherent states with positive Glauber-Sudarshan $P$-functions), the output is strictly separable and unentangled. However, if an input state is **non-classical** (such as a single photon $|1\rangle$, a Fock state $|n\rangle$, or a squeezed state), the beam splitter converts this single-mode quantumness into **inter-modal bipartite entanglement**!
> 2. **Single-Photon Non-Locality:** When a single photon encounters a 50:50 beam splitter, it does not split into two half-photons. Instead, the quantum state enters an entangled superposition $\frac{|1, 0\rangle - |0, 1\rangle}{\sqrt{2}}$. Detecting the photon at output port $a$ instantaneously collapses the field at port $b$ to vacuum $|0\rangle$, manifesting the quintessential nonlocal feature of quantum mechanics with a single quanta of light.
> 3. **The Coherent State Exception:** Coherent states $|\alpha\rangle$ are the quantum states that most closely mimic classical waves. Because $|\alpha\rangle_a |0\rangle_b$ splits into two uncorrelated coherent states $|\alpha/\sqrt{2}\rangle_a |-\alpha/\sqrt{2}\rangle_b$, the output density matrix factorizes perfectly: $\hat{\rho}_{ab} = \hat{\rho}_a \otimes \hat{\rho}_b$, yielding identically zero entanglement entropy.



In [ ]:
# Interactive Exploration: Beam Splitter Dynamics & Entanglement Tuning

def interactive_beam_splitter_explorer(input_type="Fock", fock_n=2, alpha_val=1.5, theta_deg=45.0):
    """
    Interactive exploration of output joint photon distributions and entanglement entropy.
    """
    theta_rad = np.radians(theta_deg)
    bs_engine = QuantumBeamSplitter(n_cutoff=14)
    
    if input_type == "Fock":
        psi_in = bs_engine.state_fock(int(fock_n), 0)
        state_label = f"Fock |{fock_n}, 0>"
    else:
        psi_in = bs_engine.state_coherent_vacuum(alpha_val)
        state_label = f"Coherent |alpha={alpha_val:.1f}, 0>"
        
    psi_out = bs_engine.transform_state(psi_in, theta_rad)
    P_joint = bs_engine.joint_photon_distribution(psi_out)
    entropy_val = bs_engine.entanglement_entropy(psi_out)
    
    T_val = np.cos(theta_rad)**2
    R_val = np.sin(theta_rad)**2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Subplot 1: Joint Distribution Heatmap
    im = ax1.imshow(P_joint[:8, :8], cmap="viridis", origin="lower")
    plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    ax1.set_title(f"Joint $P(n_a, n_b)$ for {state_label}\n($T={T_val:.2f}, R={R_val:.2f}$)")
    ax1.set_xlabel("Output Port $b$ Photons")
    ax1.set_ylabel("Output Port $a$ Photons")
    
    # Subplot 2: Marginal output distributions
    p_a = np.sum(P_joint, axis=1)
    p_b = np.sum(P_joint, axis=0)
    n_ax = np.arange(8)
    
    ax2.bar(n_ax - 0.15, p_a[:8], width=0.3, color="#2980b9", label="Port $a$ $P(n_a)$")
    ax2.bar(n_ax + 0.15, p_b[:8], width=0.3, color="#e67e22", label="Port $b$ $P(n_b)$")
    ax2.set_title(f"Marginal Distributions\nEntanglement Entropy $S = {entropy_val:.4f}$ nats")
    ax2.set_xlabel("Photon Number")
    ax2.set_ylabel("Probability")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

# Test invocation
interactive_beam_splitter_explorer(input_type="Fock", fock_n=3, theta_deg=45.0)

try:
    import ipywidgets as widgets
    from IPython.display import display
    w = widgets.interactive(
        interactive_beam_splitter_explorer,
        input_type=widgets.Dropdown(options=["Fock", "Coherent"], value="Fock", description="Input Type:"),
        fock_n=widgets.IntSlider(value=2, min=1, max=6, step=1, description="Fock n:"),
        alpha_val=widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.25, description=r"$\alpha$:"),
        theta_deg=widgets.FloatSlider(value=45.0, min=0.0, max=90.0, step=5.0, description=r"$\theta^\circ$:")
    )
    display(w)
except Exception:
    pass



---

## Guided Computational Challenges & Exercises

### Challenge 1: Total Photon Number Conservation
1. Prove analytically using the Schwinger $SU(2)$ representation that $[\hat{U}_{\text{BS}}(\theta), \hat{N}_{\text{tot}}] = 0$.
2. For an arbitrary input state $|\psi_{\text{in}}\rangle$ and arbitrary splitting angle $\theta$, assert numerically that:
   $$\langle \hat{N}_{\text{tot}} \rangle_{\text{out}} = \langle \hat{N}_{\text{tot}} \rangle_{\text{in}}$$
   to within precision $< 10^{-14}$.

---

### Challenge 2: Single-Photon Path Entanglement Benchmark
1. For an input single photon $|1, 0\rangle$ on a 50:50 beam splitter ($\theta = \pi/4$), show analytically that the reduced density matrix of mode $a$ is:
   $$\hat{\rho}_a = \frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1|$$
2. Prove that the entanglement entropy is $S = \ln 2 \approx 0.693147\dots \text{ nats}$.
3. Assert that the numerical engine computes $S(\hat{\rho}_a) = \ln 2$ to within an absolute error $< 10^{-12}$.

---

### Challenge 3: Coherent State Separability Benchmark
1. Demonstrate that for any coherent state amplitude $\alpha$ and any angle $\theta$, the output state is strictly separable:
   $$|\psi_{\text{out}}\rangle = |\alpha\cos\theta\rangle_a \otimes |-\alpha\sin\theta\rangle_b$$
2. Assert numerically that the entanglement entropy vanishes identically: $S(\hat{\rho}_a) < 10^{-14}$.



In [ ]:
# Numerical Verification of Challenges 1, 2, and 3: Beam Splitter Benchmarks

bs_engine = QuantumBeamSplitter(n_cutoff=18)
theta_5050 = np.pi / 4.0

# -------------------------------------------------------------
# Part 1: Total Photon Conservation (Challenge 1)
# -------------------------------------------------------------
# Test with a multi-photon superposition input
psi_test_in = (bs_engine.state_fock(3, 1) + 1j * bs_engine.state_fock(2, 2)) / np.sqrt(2.0)
N_in = float(np.real(np.dot(psi_test_in.conj().T, np.dot(bs_engine.N_tot, psi_test_in))))

# Evolve under arbitrary angle theta = 0.37 * pi
psi_test_out = bs_engine.transform_state(psi_test_in, 0.37 * np.pi)
N_out = float(np.real(np.dot(psi_test_out.conj().T, np.dot(bs_engine.N_tot, psi_test_out))))

delta_N = np.abs(N_out - N_in)
print(f"Total Photon Number Conservation Check:")
print(f"- Input <N_tot>:  {N_in:.6f}")
print(f"- Output <N_tot>: {N_out:.6f}")
print(f"- Deviation:      {delta_N:.2e}")
assert delta_N < 1e-13, f"Photon number conservation violated: dev = {delta_N:.2e} >= 1e-13!"

# -------------------------------------------------------------
# Part 2: Single-Photon Path Entanglement S = ln(2) (Challenge 2)
# -------------------------------------------------------------
psi_single = bs_engine.state_fock(1, 0)
psi_single_out = bs_engine.transform_state(psi_single, theta_5050)
S_single = bs_engine.entanglement_entropy(psi_single_out)
expected_S_single = np.log(2.0)

err_single = np.abs(S_single - expected_S_single)
print(f"\nSingle-Photon Path Entanglement Benchmark:")
print(f"- Numerical S:  {S_single:.12f} nats")
print(f"- Expected ln2: {expected_S_single:.12f} nats")
print(f"- Error:        {err_single:.2e}")
assert np.isclose(S_single, expected_S_single, atol=1e-12), f"Single-photon entropy mismatch: {err_single} >= 1e-12!"

# -------------------------------------------------------------
# Part 3: Coherent State Separability S = 0 (Challenge 3)
# -------------------------------------------------------------
psi_coh = bs_engine.state_coherent_vacuum(alpha=1.2)
psi_coh_out = bs_engine.transform_state(psi_coh, theta_5050)
S_coh = bs_engine.entanglement_entropy(psi_coh_out)

print(f"\nCoherent State Separability Benchmark (|alpha=1.2, 0>):")
print(f"- Numerical S: {S_coh:.2e} nats (Expected: 0.0)")
assert S_coh < 1e-11, f"Coherent state generated non-zero entanglement: S = {S_coh:.2e} >= 1e-11!"

# -------------------------------------------------------------
# Part 4: Binomial Distribution Verification for Fock |4, 0>
# -------------------------------------------------------------
psi_fock4 = bs_engine.state_fock(4, 0)
psi_fock4_out = bs_engine.transform_state(psi_fock4, theta_5050)
P_joint4 = bs_engine.joint_photon_distribution(psi_fock4_out)
P_a_emp = np.sum(P_joint4, axis=1)[:5]

k_arr = np.arange(5)
P_a_expected = binom(4, k_arr) * (0.5**4)
max_binom_err = np.max(np.abs(P_a_emp - P_a_expected))
print(f"\nBinomial Distribution Benchmark (|4, 0> input):")
print(f"- Max deviation from B(4, 0.5): {max_binom_err:.2e}")
assert max_binom_err < 1e-12, f"Binomial distribution mismatch: {max_binom_err} >= 1e-12!"

print("=" * 70)
print("SUCCESS: Quantum Beam Splitter Numerically Verified & Benchmarked!")
print(f"- Photon conservation: |Delta N| = {delta_N:.2e} (PASS: < 1e-13)")
print(f"- Path Bell state:     S = {S_single:.6f} nats == ln(2) (PASS: < 1e-12)")
print(f"- Coherent state:      S = {S_coh:.2e} nats == 0 (PASS: < 1e-12)")
print(f"- Binomial statistics: Max err = {max_binom_err:.2e} (PASS: < 1e-12)")
print("=" * 70)



---

## Summary & Next Steps

### Key Takeaways from Notebook 5.1
1. **Linear Symplectic Transformation:** A quantum beam splitter performs an $SU(2)$ rotation on input mode operators: $\hat{a}_{\text{out}} = \hat{a}\cos\theta + \hat{b}\sin\theta$.
2. **Total Photon Number Invariant:** The generator commutes with the total photon number $[\hat{U}_{\text{BS}}, \hat{N}_{\text{tot}}] = 0$, guaranteeing strict photon number conservation across all scattering events.
3. **Classical vs. Non-Classical Inputs:** Coherent states $|\alpha\rangle|0\rangle$ split into unentangled product states $|\alpha/\sqrt{2}\rangle|-\alpha/\sqrt{2}\rangle$.
4. **Path Entanglement:** Non-classical inputs (e.g., single photon $|1, 0\rangle$) produce maximally entangled path states $\frac{|1, 0\rangle - |0, 1\rangle}{\sqrt{2}}$ with von Neumann entropy $S = \ln 2$.

---

### Up Next: Notebook 5.2
In [Notebook 5.2: Two-Photon Interference & The Hong-Ou-Mandel Effect](02_hong_ou_mandel_interference.ipynb), we introduce **two indistinguishable photons** arriving simultaneously at the beam splitter input ports $|1, 1\rangle$, observing quantum destructive interference, photon bunching, and the celebrated Hong-Ou-Mandel dip.

